In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torchMPC.mpc import mpc
from torchMPC.mpc import util
from torchMPC.mpc.env_dx import cartpole
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pdb
import re

for i in range(1):
    pass
print('GOGOGO!')

In [ ]:
Q = torch.tensor([[0.6009, 0.2566, 0.7936, 0.9408, 0.1332],
        [0.9346, 0.5936, 0.8694, 0.5677, 0.7411],
        [0.4294, 0.8854, 0.5739, 0.2666, 0.6274],
        [0.2696, 0.4414, 0.2969, 0.8317, 0.1053],
        [0.2695, 0.3588, 0.1994, 0.5472, 0.0062]])
print(Q.T @ Q)
eigenvalues = torch.linalg.eigvalsh(Q.T @ Q)
print(eigenvalues)

In [ ]:
# Square F
class mpcTorchCost(nn.Module):
    def __init__(self, n_state, n_ctrl, tau0):
        super().__init__()
        self.n_state = n_state
        self.n_ctrl = n_ctrl

        # Equilibrium point. 注册为buffer，可以跟着模型一起加载到GPU上
        if isinstance(tau0, torch.Tensor):
            self.register_buffer('tau0', tau0)
        else:
            self.register_buffer('tau0', torch.tensor(tau0))

        self.q = nn.Parameter(torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0]))
        self.f = nn.Parameter(torch.randn(n_state, n_state))

        self.register_buffer('R', torch.eye(n_ctrl) * 0.1)

    def forward(self, tau, terminal=False):
        """
        Args:
            tau: [batch_size, n_state + n_ctrl], concatenated form of state and action
            t: Current timestamp
            T: Total prediction horizon
        Returns:
            cost: [batch_size] 
        """
        batch_size = tau.size(0)
        
        # 对于车杆问题，期望渐进稳定到顶点，所以需要减去参考点
        # 控制的参考点就是0
        state = tau[:, :self.n_state] - self.tau0[:self.n_state] 
        ctrl = tau[:, self.n_state:]
        
        # 如果是最后一个时间步，只计算终端损失
        if terminal:
            F_postive = (self.f.T @ self.f).repeat(batch_size, 1, 1)
            return 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), F_postive), state.unsqueeze(2)).squeeze(-1).squeeze(-1) # Return shape: (batch_size, )
        else:
            return self.running_cost(state, ctrl)
    
    def running_cost(self, state, ctrl): 
        # 简单二次型运行代价
        batch_size = state.shape[0]

        Q_postive = torch.diag(self.q).pow(2).repeat(batch_size, 1, 1)
        xTQx = 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), Q_postive), state.unsqueeze(2))
        uTRu = 0.5 * torch.bmm(torch.bmm(ctrl.unsqueeze(1), self.R.repeat(batch_size, 1, 1)), ctrl.unsqueeze(2))
        return (xTQx + uTRu).squeeze(-1).squeeze(-1) # Return shape: (batch_size,)

In [33]:
# Diag F
# costNet running and terminal cost
class mpcTorchCost1(nn.Module):
    def __init__(self, n_state, n_ctrl, tau0):
        super().__init__()
        self.n_state = n_state
        self.n_ctrl = n_ctrl

        # Equilibrium point. 注册为buffer，可以跟着模型一起加载到GPU上
        if isinstance(tau0, torch.Tensor):
            self.register_buffer('tau0', tau0)
        else:
            self.register_buffer('tau0', torch.tensor(tau0))

        self.q = nn.Parameter(torch.rand(n_state))
        self.f = nn.Parameter(torch.rand(n_state))

        # self.register_buffer('q', torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0]))
        self.register_buffer('R', torch.eye(n_ctrl) * 0.001)

    def forward(self, tau, terminal=False):
        """
        Args:
            tau: [batch_size, n_state + n_ctrl], concatenated form of state and action
            t: Current timestamp
            T: Total prediction horizon
        Returns:
            cost: [batch_size] 
        """
        batch_size = tau.size(0)
        
        # 对于车杆问题，期望渐进稳定到顶点，所以需要减去参考点
        # 控制的参考点就是0
        state = tau[:, :self.n_state] - self.tau0[:self.n_state] 
        ctrl = tau[:, self.n_state:]
        
        # 如果是最后一个时间步，只计算终端损失
        if terminal:
            # F_postive = (self.f.T @ self.f).repeat(batch_size, 1, 1)
            F_postive = torch.diag(self.f).pow(2).repeat(batch_size, 1, 1)
            return 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), F_postive), state.unsqueeze(2)).squeeze(-1).squeeze(-1) # Return shape: (batch_size, )
        else:
            return self.running_cost(state, ctrl)
    
    def running_cost(self, state, ctrl): 
        # 简单二次型运行代价
        batch_size = state.shape[0]

        Q_postive = torch.diag(self.q).pow(2).repeat(batch_size, 1, 1)
        xTQx = 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), Q_postive), state.unsqueeze(2))
        uTRu = 0.5 * torch.bmm(torch.bmm(ctrl.unsqueeze(1), self.R.repeat(batch_size, 1, 1)), ctrl.unsqueeze(2))
        return (xTQx + uTRu).squeeze(-1).squeeze(-1) # Return shape: (batch_size,)

In [34]:
# 均匀分布采样
def uniform(shape, low, high):
    r = high - low
    return torch.rand(shape) * r + low

# 生成初始状态，来自mpc torch
def cartpole_initx(n_batch, angle=180):
    ratio = angle / 180.0
    th = uniform(n_batch, -ratio*np.pi, ratio*np.pi)
    thdot = uniform(n_batch, -.5 * ratio, .5 * ratio)
    x = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xdot = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
    ref = torch.tensor([0., 0., 1., 0., 0.])
    return xinit 

# 可视化

In [35]:
device = 'cpu'

n_batch, T, mpc_T = 64, 200, 25

dx = cartpole.CartpoleDx()
t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

equilibrium = dx.goal_state.to(device)
cost = mpcTorchCost1(dx.n_state, dx.n_ctrl, equilibrium).to(device)

model_path = 'testCartPole/2505171_Diag_QF_seed1_T25_FullRange/pth/199.pth'
cost.load_state_dict(torch.load(model_path))
# cost.q = nn.Parameter(torch.tensor([ 0.6614,  0.2669,  0.0617,  0.6213, -0.4519]))
# cost.f = nn.Parameter(torch.tensor([-0.1661, -1.5228,  0.3817, -1.0276, -0.5631]))
# cost.f = nn.Parameter(torch.tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
#         [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
#         [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
#         [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
#         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]))
print(f'Q: {cost.q.pow(2)}')
print(f'F: {cost.f.pow(2)}')

x = cartpole_initx(n_batch).to(device)
u_init = None
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        n_batch=n_batch,
        u_init=u_init,
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=mpc.GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, cost, dx)

    next_action = nominal_actions[0]
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 因为u[-1]本来就是0了

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    n_col = 4
    n_row = n_batch // n_col
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        dx.get_frame(x[i], ax=axs[i])
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    plt.close(fig)

C:\Users\90534\AppData\Local\Temp\ipykernel_14212\2534149039.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cost.load_state_dict(torch.load(model_path))


Q: tensor([4.0028e-09, 8.7750e-01, 7.1915e-08, 4.5031e-02, 1.8427e-06],
       grad_fn=<PowBackward0>)
F: tensor([0.0030, 0.1152, 0.1153, 0.9860, 0.1663], grad_fn=<PowBackward0>)


100%|██████████| 200/200 [04:42<00:00,  1.41s/it]


# 条件判断稳定

In [ ]:
def stable_num(state, u, eps=1e-2):
    batch_size = state.shape[0]
    refx = torch.tensor([0., 0., 1., 0., 0.]).unsqueeze(0).repeat(batch_size, 1)
    refu = torch.tensor([0.]).unsqueeze(0).repeat(batch_size, 1)
    whether_stable = torch.all((state - refx) < eps, dim=1) * torch.all((u - refu) < eps, dim=1)
    return torch.sum(whether_stable).item()

# test_states = cartpole_initx(10)
# test_states[0] = torch.tensor([0., 0., 1., 0., 0.])
# test_states[1] = torch.tensor([0., 0., 1., 0., 0.])
# test_states[2] = torch.tensor([0., 0., 1., 0., 0.])
# test_controls = torch.randn(10, 1) * 0.1
# test_controls[0] = torch.tensor([0.])
# test_controls[1] = torch.tensor([0.])
# test_controls[2] = torch.tensor([0.])
# print(stable_num(test_states, test_controls))

In [ ]:
device = 'cpu'

n_batch, T, mpc_T = 128, 150, 32

dx = cartpole.CartpoleDx()
t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

equilibrium = dx.goal_state.to(device)
cost = mpcTorchCost(dx.n_state, dx.n_ctrl, equilibrium).to(device)

model_path = 'D:\Docs\code_lib\graduation_test\testCartPole\250515Train_GO!\pth\77.pth'
cost.load_state_dict(torch.load(model_path))

x = cartpole_initx(n_batch)
u_init = None
x_list = []
u_list = []
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        n_batch=n_batch,
        u_init=u_init,
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=mpc.GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, cost, dx)

    next_action = nominal_actions[0]
    x_list.append(x)
    u_list.append(next_action)
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 因为u[-1]本来就是0了

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    # n_col = 4
    # n_row = n_batch // n_col
    # fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    # axs = axs.reshape(-1)
    # for i in range(n_batch):
    #     dx.get_frame(x[i], ax=axs[i])
    #     axs[i].get_xaxis().set_visible(False)
    #     axs[i].get_yaxis().set_visible(False)
    # fig.tight_layout()
    # fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    # plt.close(fig)

In [ ]:
print(x_list[-1])
print(stable_num(x_list[-1], u_list[-1], eps=1e-2))

In [ ]:
print(x_list[-1])
print(stable_num(x_list[-1], u_list[-1], eps=1e-2))